# Dataset Standardization for Neuro-Symbolic Reasoning Evaluation

This notebook demonstrates the dataset standardization process for ontological type-constrained abductive completion in neuro-symbolic reasoning pipelines.

## What this artifact does

This script standardizes multiple reasoning datasets into a unified JSON schema for evaluating neuro-symbolic reasoning systems. The datasets span:
- **EntailmentBank** (Task 2 & 3): Science question answering with entailment trees
- **MuSR**: Multiple-choice reasoning in narrative contexts (murder mystery, object placements)
- **ProofWriter**: Logical reasoning with proof structures

The output is a standardized JSON file with input-output pairs and metadata fields.

In [ ]:
# Install dependencies - follows aii-colab pattern for Colab + local compatibility
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is NOT pre-installed on Colab, always install
_pip('loguru')
_pip('requests')

# Core packages pre-installed on Colab - install locally to match versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
# Standard imports from the original script
import json
import random
import re
import sys
import time
from pathlib import Path

import requests
from loguru import logger

# Setup logger (original configuration)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

print("Imports complete")

In [ ]:
# Data loading helper - GitHub URL with local fallback pattern
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6bf5ea-ontological-type-constrained-abductive-c/main/iter_1/gen_art_dataset_1/demo/mini_demo_data.json"

def load_data():
    """Load data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed: {e}")
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

import os
data = load_data()
print(f"Loaded data with {data['metadata']['total_examples']} examples from {data['metadata']['n_datasets']} datasets")

In [ ]:
# Config cell - ALL tunable parameters set to ABSOLUTE MINIMUM values
# These can be increased gradually for scaling (see TODO 5)

# Dataset caps (original: PROOFWRITER_CAP=5000, RULETAKER_CAP=3000)
PROOFWRITER_CAP = 5  # MINIMUM: just 5 examples
RULETAKER_CAP = 5    # MINIMUM: just 5 examples

# Span verifier holdout (original: n_docs=100)
SPAN_HOLDOUT_DOCS = 2  # MINIMUM: just 2 docs

# ConceptNet index parameters (original: limit=100, max_per_term=15)
CN_TERMS_LIMIT = 5     # MINIMUM: just 5 terms
CN_MAX_PER_TERM = 3    # MINIMUM: just 3 relations per term

# Set random seed for reproducibility
random.seed(42)

print("Config set:")
print(f"  PROOFWRITER_CAP = {PROOFWRITER_CAP}")
print(f"  RULETAKER_CAP = {RULETAKER_CAP}")
print(f"  SPAN_HOLDOUT_DOCS = {SPAN_HOLDOUT_DOCS}")
print(f"  CN_TERMS_LIMIT = {CN_TERMS_LIMIT}")
print(f"  CN_MAX_PER_TERM = {CN_MAX_PER_TERM}")

## Helper Functions

These functions parse the dataset-specific formats into a unified schema:
- `parse_eb_context`: Parses EntailmentBank context strings into sentence dictionaries
- `parse_eb_proof_leaves`: Extracts leaf sentence IDs from proof strings

In [ ]:
# ─────────────────────────────────────────────
# Helpers (from original data.py)
# ─────────────────────────────────────────────

def parse_eb_context(context_str: str) -> dict[str, str]:
    """Parse 'sent1: text sent2: text ...' into {sent_id: text}."""
    result = {}
    pattern = re.compile(r'(sent\d+):\s*')
    parts = pattern.split(context_str)
    # parts = ['', 'sent1', 'text1 ', 'sent2', 'text2 ', ...]
    for i in range(1, len(parts) - 1, 2):
        sent_id = parts[i].strip()
        text = parts[i + 1].strip()
        result[sent_id] = text
    return result


def parse_eb_proof_leaves(proof_str: str) -> list[str]:
    """Extract leaf sentence IDs from proof string.
    e.g. 'sent10 & sent12 & sent4 -> hypothesis;' -> ['sent10', 'sent12', 'sent4']
    Also handles multi-step proofs: 'sent1 & int1 -> int2; sent3 & int2 -> hypothesis;'
    Leaves are sent IDs that appear on the left of any arrow but are NOT intermediate nodes.
    """
    intermediate_ids = set()
    all_left_ids: list[str] = []
    for step in proof_str.split(";"):
        step = step.strip()
        if "->" not in step:
            continue
        left, right = step.split("->", 1)
        right_id = right.strip()
        if right_id.startswith("int"):
            intermediate_ids.add(right_id)
        tokens = re.findall(r'sent\d+|int\d+', left)
        all_left_ids.extend(tokens)
    # leaves = sent IDs on the left side (not int IDs)
    leaves = [t for t in all_left_ids if t.startswith("sent")]
    return list(dict.fromkeys(leaves))  # deduplicate preserving order

print("Helper functions defined")

# Test the helpers
test_ctx = "sent1: Earth rotates | sent2: sun rises"
print(f"parse_eb_context test: {parse_eb_context(test_ctx)}")

test_proof = "sent1 & sent2 -> hypothesis;"
print(f"parse_eb_proof_leaves test: {parse_eb_proof_leaves(test_proof)}")

## Dataset Loaders

These functions load and standardize each dataset into the unified schema.
For the demo, we'll use the mini_demo_data that was already loaded.

In [ ]:
# ─────────────────────────────────────────────
# Process the loaded mini_demo_data
# ─────────────────────────────────────────────

# The mini_demo_data is already in the standardized format
# Just extract and display the examples

datasets = []
for ds in data['datasets']:
    datasets.append({"dataset": ds['dataset'], "examples": ds['examples']})
    print(f"Loaded {ds['dataset']}: {len(ds['examples'])} examples")

# Display first example from each dataset
print("\n=== Sample Examples ===")
for ds in datasets:
    print(f"\nDataset: {ds['dataset']}")
    if ds['examples']:
        ex = ds['examples'][0]
        print(f"  Input: {ex['input'][:100]}...")
        print(f"  Output: {ex['output']}")
        print(f"  Task type: {ex.get('metadata_task_type', 'N/A')}")

## Build Span Verifier Holdout

Creates a holdout set for verifying sentence spans in entailment trees.
Generates genuine and negative pairs from EntailmentBank context sentences.

In [ ]:
# ─────────────────────────────────────────────
# Build span verifier holdout (simplified for demo)
# ─────────────────────────────────────────────

def build_span_verifier_holdout_demo(n_docs: int = 2) -> list[dict]:
    """Build span verifier holdout from the loaded data."""
    # Use entailment_bank_task2 data if available
    eb_data = None
    for ds in data['datasets']:
        if 'entailment_bank' in ds['dataset']:
            eb_data = ds['examples']
            break
    
    if not eb_data:
        print("No entailment bank data found, skipping span verifier")
        return []
    
    # For demo, create simple pairs from context
    pairs = []
    for i, ex in enumerate(eb_data[:n_docs]):
        ctx_map = parse_eb_context(ex.get('input', ''))
        sent_ids = list(ctx_map.keys())
        if len(sent_ids) >= 2:
            # Genuine pair
            pairs.append({
                "doc_id": ex.get('metadata_id', f'doc_{i}'),
                "predicate_text": ctx_map[sent_ids[0]],
                "claimed_span": ctx_map[sent_ids[0]],
                "is_genuine": True,
                "entailment_tree_leaf_id": sent_ids[0],
            })
            # Negative pair
            pairs.append({
                "doc_id": ex.get('metadata_id', f'doc_{i}'),
                "predicate_text": ctx_map[sent_ids[0]],
                "claimed_span": ctx_map[sent_ids[1]],
                "is_genuine": False,
                "entailment_tree_leaf_id": sent_ids[0],
            })
    
    print(f"Built {len(pairs)} span verifier pairs ({sum(1 for p in pairs if p['is_genuine'])} genuine)")
    return pairs

span_pairs = build_span_verifier_holdout_demo(n_docs=SPAN_HOLDOUT_DOCS)
if span_pairs:
    print(f"\nSample pair (genuine): {span_pairs[0]}")

## ConceptNet Index (Demo Version)

Builds a ConceptNet lookup index for terms extracted from the datasets.
For the demo, we use a simplified version that doesn't require API calls.

In [ ]:
# ─────────────────────────────────────────────
# ConceptNet Index (demo version - no API calls)
# ─────────────────────────────────────────────

def build_conceptnet_index_demo(terms: list[str], relations: list[str], max_per_term: int = 3) -> dict:
    """Demo version - creates a mock ConceptNet index without API calls."""
    index = {}
    for term in terms[:CN_TERMS_LIMIT]:
        # Create mock edges for demo
        edges = []
        for i, rel in enumerate(relations[:max_per_term]):
            edges.append({
                "relation": rel,
                "start": term,
                "end": f"related_{i}",
                "weight": 1.0,
            })
        index[term] = edges
    print(f"Built demo ConceptNet index with {len(index)} terms")
    return index

CN_RELATIONS = [
    "IsA", "UsedFor", "CapableOf", "HasProperty", "Causes", "RelatedTo", "PartOf",
    "HasPrerequisite", "CausesDesire", "CreatedBy", "DefinedAs", "HasA",
]

# Extract terms from data (simplified)
def extract_terms_demo(limit: int = 5) -> list[str]:
    """Extract sample terms from the loaded data."""
    terms = set()
    for ds in data['datasets']:
        for ex in ds['examples'][:3]:
            words = re.findall(r'\b[a-z]{5,}\b', ex.get('input', '').lower())
            terms.update(words[:3])
    return list(terms)[:limit]

terms = extract_terms_demo(limit=CN_TERMS_LIMIT)
print(f"Extracted terms: {terms}")

cn_index = build_conceptnet_index_demo(terms, CN_RELATIONS, max_per_term=CN_MAX_PER_TERM)
print(f"\nSample entry: {list(cn_index.items())[0] if cn_index else 'empty'}")

## Summary and Output

Compile the results and display summary statistics.

In [ ]:
# ─────────────────────────────────────────────
# Summary statistics and output
# ─────────────────────────────────────────────

# Calculate totals
total_examples = sum(len(ds['examples']) for ds in datasets)

print("=== Dataset Summary ===")
print(f"Total examples: {total_examples}")
print(f"Number of datasets: {len(datasets)}")
for ds in datasets:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")

print(f"\nSpan verifier pairs: {len(span_pairs)}")
print(f"ConceptNet terms indexed: {len(cn_index)}")

# Create output structure (similar to full_data_out.json)
output = {
    "datasets": datasets,
    "metadata": {
        "total_examples": total_examples,
        "n_datasets": len(datasets),
        "datasets_included": [ds['dataset'] for ds in datasets],
        "span_verifier_n_pairs": len(span_pairs),
        "conceptnet_n_terms": len(cn_index),
        "demo_mode": True,
    },
}

print("\n=== Output Preview ===")
print(json.dumps(output['metadata'], indent=2))

In [ ]:
# Visualization: Display dataset statistics
import matplotlib.pyplot as plt

# Prepare data for visualization
ds_names = [ds['dataset'] for ds in datasets]
ds_sizes = [len(ds['examples']) for ds in datasets]

# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart of dataset sizes
ax1 = axes[0]
bars = ax1.bar(ds_names, ds_sizes, color='skyblue', edgecolor='navy')
ax1.set_xlabel('Dataset')
ax1.set_ylabel('Number of Examples')
ax1.set_title('Dataset Sizes')
ax1.tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar, size in zip(bars, ds_sizes):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{size}', ha='center', va='bottom')

# Pie chart of dataset distribution
ax2 = axes[1]
if sum(ds_sizes) > 0:
    ax2.pie(ds_sizes, labels=ds_names, autopct='%1.1f%%', startangle=90)
    ax2.set_title('Dataset Distribution')

plt.tight_layout()
plt.show()

# Print summary table
print("\n=== Summary Table ===")
print(f"{'Dataset':<30} {'Examples':>10} {'% of Total':>12}")
print("-" * 55)
for ds, size in zip(ds_names, ds_sizes):
    pct = (size / total_examples * 100) if total_examples > 0 else 0
    print(f"{ds:<30} {size:>10} {pct:>11.1f}%")
print("-" * 55)
print(f"{'TOTAL':<30} {total_examples:>10} {'100.0%':>12}")